# 02 — Build `claims_fe` Wheel

Materializes the `claims_fe` source tree from inline strings, builds
`claims_fe-0.1.0-py3-none-any.whl`, and uploads it to a UC Volume for use by the
Model Serving pyfunc (notebook 03).

Wheel contents:
- `claims_fe.transformer.ClaimFeatureTransformer` — the `mlflow.pyfunc.PythonModel`
- `claims_fe.lexicons` — attorney / SIU / medical / subrogation / urgency / health dictionaries
- `claims_fe.features.{structural, temporal, text_flags}` — pure-function extractors

Pure `pandas` + `numpy` only. No embeddings, no LLM — v1 is SLA-safe regex/dict logic.

In [0]:
%pip install build
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
CATALOG = "fins_genai"
SCHEMA = "classic_ml"
WHEEL_VOLUME = "claims_fe_wheels"
WHEEL_VERSION = "0.1.0"

STAGING_DIR = "/tmp/claims_fe_build"
DIST_DIR = "/tmp/claims_fe_dist"
WHEEL_VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{WHEEL_VOLUME}"

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{WHEEL_VOLUME}")

DataFrame[]

## Wheel source — written inline

Each entry in `SOURCES` below is one file in the wheel. They're materialized to
`STAGING_DIR` in the next cell.

In [0]:
PYPROJECT = f'''\
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "claims_fe"
version = "{WHEEL_VERSION}"
description = "Claim feature-engineering pyfunc — regex/dict-based extractors for insurance claim payloads."
requires-python = ">=3.10"
dependencies = [
    "pandas>=1.5",
    "numpy>=1.23",
]

[tool.setuptools.packages.find]
include = ["claims_fe*"]
'''

INIT_PY = f'''\
"""claims_fe — feature engineering for insurance claim payloads."""
from claims_fe.transformer import ClaimFeatureTransformer

__version__ = "{WHEEL_VERSION}"
__all__ = ["ClaimFeatureTransformer"]
'''

LEXICONS_PY = '''\
"""Module-level lexicon constants. Loaded once; no file I/O per call."""
import re

# Attorney / legal representation
ATTORNEY_PATTERNS = [
    r"\\bretained counsel\\b",
    r"\\bretain counsel\\b",
    r"\\bletter of representation\\b",
    r"\\bLOR\\b",
    r"\\bcease\\s+(?:and\\s+desist|communication|contact)\\b",
    r"\\battorney\\s+(?:representing|for the)\\b",
    r"\\bmy attorney\\b",
    r"\\blaw firm\\b",
    r"\\bdemand letter\\b",
    r"\\brepresented by\\b",
    r"\\bour client'?s?\\b",
    r"\\bbodily injury claim\\b",
]

# SIU red flags
SIU_PATTERNS = [
    r"\\bno police report\\b",
    r"\\bunable to locate\\b",
    r"\\bcash[-\\s]only\\b",
    r"\\bno receipt\\b",
    r"\\bno witness\\b",
    r"\\bwitness unavailable\\b",
    r"\\bprior claim\\b",
    r"\\brecent policy\\b",
    r"\\bstaged[-\\s]loss\\b",
    r"\\bstaged\\s+loss\\b",
    r"\\bsuspicious\\b",
    r"\\bunder investigation\\b",
]

# Medical severity
MEDICAL_PATTERNS = [
    r"\\bsurgery\\b",
    r"\\bsurgical\\b",
    r"\\bMRI\\b",
    r"\\bCT scan\\b",
    r"\\bER (?:visit)?\\b",
    r"\\bemergency room\\b",
    r"\\bphysical therapy\\b",
    r"\\bchiropractor\\b",
    r"\\bpermanent (?:injury|disability)\\b",
    r"\\bhospitalization\\b",
    r"\\binpatient\\b",
    r"\\borthopedic\\b",
]

# Subrogation opportunity
SUBROGATION_PATTERNS = [
    r"\\bthird[-\\s]party\\b",
    r"\\bat fault\\b",
    r"\\bother driver\\b",
    r"\\bnegligent\\b",
    r"\\bdefective\\b",
    r"\\bmanufacturer\\b",
    r"\\bproduct liability\\b",
    r"\\bsubrogation\\b",
]

# Urgency
URGENCY_WORDS = ["URGENT", "ASAP", "IMMEDIATELY", "EXPEDITE", "TIME-SENSITIVE"]

# Health disclosures (PHI)
HEALTH_PATTERNS = [
    r"\\bcancer\\b",
    r"\\bchemotherapy\\b",
    r"\\bimmunocompromised\\b",
    r"\\bpregnant\\b",
    r"\\bdisabled?\\b",
    r"\\bdisability\\b",
    r"\\bmedication\\b",
    r"\\bdiabetes\\b",
]

DOLLAR_PATTERN = r"\\$\\s?(\\d{1,3}(?:,\\d{3})*(?:\\.\\d{2})?)"


def _compile_all(patterns):
    return [re.compile(p, re.IGNORECASE) for p in patterns]


COMPILED = {
    "attorney":    _compile_all(ATTORNEY_PATTERNS),
    "siu":         _compile_all(SIU_PATTERNS),
    "medical":     _compile_all(MEDICAL_PATTERNS),
    "subrogation": _compile_all(SUBROGATION_PATTERNS),
    "health":      _compile_all(HEALTH_PATTERNS),
    "dollar":      re.compile(DOLLAR_PATTERN),
}
'''

FEATURES_INIT_PY = '''\
"""Feature extractor subpackage. Each module is a pure-function set."""
'''

FEATURES_STRUCTURAL_PY = '''\
"""Length / exposure / passthrough features."""


def structural_features(claim: dict) -> dict:
    docs = claim.get("ConcatenatedDocs") or ""
    notes = claim.get("ConcatenatedNotes") or ""
    incidents = claim.get("ConcatenatedIncidents") or ""
    desc = claim.get("CLM_DESC_TXT") or ""

    exposures = claim.get("Exposures") or []
    exposure_types = [e.get("EXPSR_TYP_CD") for e in exposures if e.get("EXPSR_TYP_CD")]

    return {
        "desc_char_count": len(desc),
        "docs_char_count": len(docs),
        "notes_char_count": len(notes),
        "incidents_char_count": len(incidents),
        "n_exposures": len(exposures),
        "has_dwelling": "Dwelling" in exposure_types,
        "has_contents": "Content" in exposure_types,
        "has_living_expenses": "LivingExpenses" in exposure_types,
        "exposure_types": exposure_types,
        "lob_type": claim.get("CLM_LOB_TYP_CD"),
        "loss_cause": claim.get("LOSS_CSE_TYP_CD"),
        "sub_type": claim.get("CLM_SBTYP_CD"),
        "state": claim.get("PLCY_STATE"),
        "jurisdiction_state": claim.get("JURISDCTN_STATE_TYP_CD"),
        "water_source": claim.get("WATER_SRC_TYP_CD"),
        "claim_id": claim.get("NK_CLM_ID"),
        "claim_num": claim.get("CLM_NUM"),
    }
'''

FEATURES_TEMPORAL_PY = '''\
"""Temporal features derived from claim dates."""
from datetime import datetime


def _parse(dttm: str):
    if not dttm:
        return None
    # Handle the trailing ".0000000" style seen in Guidewire exports
    if "." in dttm:
        head, _, _tail = dttm.partition(".")
        dttm = head
    try:
        return datetime.strptime(dttm, "%Y-%m-%dT%H:%M:%S")
    except ValueError:
        return None


def temporal_features(claim: dict) -> dict:
    loss = _parse(claim.get("CLM_LOSS_DTTM"))
    reported = _parse(claim.get("CLM_RPRTD_DTTM"))

    days_loss_to_report = None
    if loss and reported:
        days_loss_to_report = (reported - loss).days

    construction_year = claim.get("PRPTY_CNSTRCTN_YR")
    property_age_at_loss = None
    if loss and isinstance(construction_year, int) and construction_year > 1800:
        property_age_at_loss = loss.year - construction_year

    return {
        "loss_date": loss.date().isoformat() if loss else None,
        "reported_date": reported.date().isoformat() if reported else None,
        "days_loss_to_report": days_loss_to_report,
        "property_age_at_loss": property_age_at_loss,
    }
'''

FEATURES_TEXT_FLAGS_PY = '''\
"""Regex/dictionary-based text feature extractors."""
import re
from claims_fe.lexicons import COMPILED, URGENCY_WORDS


def _count_matches(text: str, compiled_list) -> int:
    if not text:
        return 0
    return sum(1 for r in compiled_list if r.search(text))


def _count_all_matches(text: str, compiled_list) -> int:
    """Total match count across all patterns (not just number of patterns that hit)."""
    if not text:
        return 0
    total = 0
    for r in compiled_list:
        total += len(r.findall(text))
    return total


def _urgency_score(text: str) -> float:
    if not text:
        return 0.0
    length = max(len(text), 1)
    urgent_hits = sum(text.count(w) for w in URGENCY_WORDS)
    exclamation_count = text.count("!")
    # Caps ratio over alphabetic characters only
    alpha = [c for c in text if c.isalpha()]
    caps_ratio = (sum(1 for c in alpha if c.isupper()) / len(alpha)) if alpha else 0.0

    score = min(
        1.0,
        urgent_hits * 0.15
        + (exclamation_count / (length / 1000)) * 0.01
        + caps_ratio * 0.3,
    )
    return round(float(score), 4)


def _dollar_amounts(text: str):
    if not text:
        return []
    matches = COMPILED["dollar"].findall(text)
    amounts = []
    for m in matches:
        try:
            amounts.append(float(m.replace(",", "")))
        except ValueError:
            continue
    return amounts


def text_flag_features(claim: dict) -> dict:
    # Combine all text sources (notes carry most signal; incidents + docs are shorter)
    notes = claim.get("ConcatenatedNotes") or ""
    docs = claim.get("ConcatenatedDocs") or ""
    incidents = claim.get("ConcatenatedIncidents") or ""
    combined = " ".join(filter(None, [notes, incidents, docs]))

    dollar_amounts = _dollar_amounts(combined)

    return {
        "attorney_mentioned": _count_matches(combined, COMPILED["attorney"]) > 0,
        "attorney_phrase_count": _count_all_matches(combined, COMPILED["attorney"]),
        "siu_red_flag_count": _count_matches(combined, COMPILED["siu"]),
        "medical_severity_flag": _count_matches(combined, COMPILED["medical"]) > 0,
        "subrogation_opportunity_flag": _count_matches(combined, COMPILED["subrogation"]) > 0,
        "health_disclosure_flag": _count_matches(combined, COMPILED["health"]) > 0,
        "urgency_score": _urgency_score(combined),
        "max_dollar_amount_mentioned": max(dollar_amounts) if dollar_amounts else None,
        "dollar_amount_count": len(dollar_amounts),
    }
'''

TRANSFORMER_PY = f'''\
"""ClaimFeatureTransformer — MLflow pyfunc wrapper around the claims_fe extractors."""
import json

import mlflow.pyfunc
import pandas as pd

from claims_fe.features.structural import structural_features
from claims_fe.features.temporal import temporal_features
from claims_fe.features.text_flags import text_flag_features


FEATURE_VERSION = "{WHEEL_VERSION}"


def _extract_one(payload_json: str) -> dict:
    errors = []
    try:
        payload = json.loads(payload_json)
    except Exception as exc:
        return {{
            "claim_id": None, "claim_num": None,
            "feature_version": FEATURE_VERSION,
            "extraction_errors": [f"json_parse_error: {{exc}}"],
        }}

    claims = payload.get("Claims") or []
    if not claims:
        return {{
            "claim_id": None, "claim_num": None,
            "feature_version": FEATURE_VERSION,
            "extraction_errors": ["no_claims_in_payload"],
        }}

    claim = claims[0]
    out = {{}}
    for fn, name in [
        (structural_features, "structural"),
        (temporal_features, "temporal"),
        (text_flag_features, "text_flags"),
    ]:
        try:
            out.update(fn(claim))
        except Exception as exc:
            errors.append(f"{{name}}_error: {{exc}}")

    out["feature_version"] = FEATURE_VERSION
    out["extraction_errors"] = errors
    return out


class ClaimFeatureTransformer(mlflow.pyfunc.PythonModel):
    """
    Pattern A feature-engineering pyfunc.

    Input : DataFrame with a single string column 'payload_json' (raw claim JSON).
    Output: DataFrame with a single string column 'features_json' (engineered features).
    """

    def load_context(self, context):
        # Nothing to load — lexicons and regex are compiled at module import.
        pass

    def predict(self, context, model_input, params=None):
        if isinstance(model_input, pd.DataFrame):
            series = model_input.iloc[:, 0]
        else:
            series = pd.Series(model_input)

        outputs = [json.dumps(_extract_one(s)) for s in series.astype(str).tolist()]
        return pd.DataFrame({{"features_json": outputs}})
'''

SOURCES = {
    "pyproject.toml":                       PYPROJECT,
    "claims_fe/__init__.py":                INIT_PY,
    "claims_fe/lexicons.py":                LEXICONS_PY,
    "claims_fe/transformer.py":             TRANSFORMER_PY,
    "claims_fe/features/__init__.py":       FEATURES_INIT_PY,
    "claims_fe/features/structural.py":     FEATURES_STRUCTURAL_PY,
    "claims_fe/features/temporal.py":       FEATURES_TEMPORAL_PY,
    "claims_fe/features/text_flags.py":     FEATURES_TEXT_FLAGS_PY,
}

## Materialize source tree & build wheel

In [0]:
import shutil
from pathlib import Path

staging = Path(STAGING_DIR)
dist = Path(DIST_DIR)
if staging.exists():
    shutil.rmtree(staging)
if dist.exists():
    shutil.rmtree(dist)
staging.mkdir(parents=True)
dist.mkdir(parents=True)

for rel_path, content in SOURCES.items():
    full = staging / rel_path
    full.parent.mkdir(parents=True, exist_ok=True)
    full.write_text(content)
    print(f"  wrote {rel_path} ({len(content)} bytes)")

  wrote pyproject.toml (393 bytes)
  wrote claims_fe/__init__.py (187 bytes)
  wrote claims_fe/lexicons.py (2260 bytes)
  wrote claims_fe/transformer.py (2123 bytes)
  wrote claims_fe/features/__init__.py (72 bytes)
  wrote claims_fe/features/structural.py (1265 bytes)
  wrote claims_fe/features/temporal.py (1158 bytes)
  wrote claims_fe/features/text_flags.py (2518 bytes)


In [0]:
%sh
cd /tmp/claims_fe_build && python -m build --wheel --outdir /tmp/claims_fe_dist 2>&1 | tail -20

copying build/lib/claims_fe/features/__init__.py -> build/bdist.linux-x86_64/wheel/./claims_fe/features
copying build/lib/claims_fe/features/structural.py -> build/bdist.linux-x86_64/wheel/./claims_fe/features
running install_egg_info
Copying claims_fe.egg-info to build/bdist.linux-x86_64/wheel/./claims_fe-0.1.0-py3.12.egg-info
running install_scripts
creating build/bdist.linux-x86_64/wheel/claims_fe-0.1.0.dist-info/WHEEL
creating '/tmp/claims_fe_dist/.tmp-2gwairrr/claims_fe-0.1.0-py3-none-any.whl' and adding 'build/bdist.linux-x86_64/wheel' to it
adding 'claims_fe/__init__.py'
adding 'claims_fe/lexicons.py'
adding 'claims_fe/transformer.py'
adding 'claims_fe/features/__init__.py'
adding 'claims_fe/features/structural.py'
adding 'claims_fe/features/temporal.py'
adding 'claims_fe/features/text_flags.py'
adding 'claims_fe-0.1.0.dist-info/METADATA'
adding 'claims_fe-0.1.0.dist-info/WHEEL'
adding 'claims_fe-0.1.0.dist-info/top_level.txt'
adding 'claims_fe-0.1.0.dist-info/RECORD'
removing b

In [0]:
wheel_files = list(dist.glob("*.whl"))
assert len(wheel_files) == 1, f"expected 1 wheel, found {wheel_files}"
wheel_src = wheel_files[0]
print(f"Built: {wheel_src} ({wheel_src.stat().st_size:,} bytes)")

Built: /tmp/claims_fe_dist/claims_fe-0.1.0-py3-none-any.whl (6,188 bytes)


## Upload wheel to UC Volume

In [0]:
import os

os.makedirs(WHEEL_VOLUME_PATH, exist_ok=True)
wheel_dst = Path(WHEEL_VOLUME_PATH) / wheel_src.name
shutil.copy(wheel_src, wheel_dst)
print(f"Uploaded: {wheel_dst}")
print(f"Volume contents:")
for f in sorted(os.listdir(WHEEL_VOLUME_PATH)):
    size = os.path.getsize(f"{WHEEL_VOLUME_PATH}/{f}")
    print(f"  {f}: {size:,} bytes")

Uploaded: /Volumes/fins_genai/classic_ml/claims_fe_wheels/claims_fe-0.1.0-py3-none-any.whl
Volume contents:
  claims_fe-0.1.0-py3-none-any.whl: 6,188 bytes


## Sanity-test the wheel

Install in the current notebook kernel, import the transformer, run it against one
synthetic payload from notebook 01, and assert the output JSON contract.

In [0]:
%pip install /tmp/claims_fe_dist/claims_fe-0.1.0-py3-none-any.whl --force-reinstall
dbutils.library.restartPython()

Processing /tmp/claims_fe_dist/claims_fe-0.1.0-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.7 MB/s eta 0:00:00
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Not uninstalling six at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-d7d05120-5ab0-4ebe-aff9-b78e4852bd7f
    Can't uninstall 'six'. No files were found to uninstall.
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Not uninstalling numpy at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-d7d05120-5ab0-4ebe-aff9-b78e4852bd7f
    Can't uninstall 'numpy'. No files were found to uninstall.
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Not uninstalling python-dateutil at /databr

In [0]:
# Re-bind constants after python restart
CATALOG = "fins_genai"
SCHEMA = "classic_ml"
TABLE_NAME = "fe_test_payloads"

In [0]:
import json
import pandas as pd
from claims_fe.transformer import ClaimFeatureTransformer

sample = (
    spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")
    .select("payload_id", "payload_json")
    .limit(1)
    .collect()[0]
)
print(f"Testing payload: {sample.payload_id}")

transformer = ClaimFeatureTransformer()
result = transformer.predict(None, pd.DataFrame({"payload_json": [sample.payload_json]}))

out = json.loads(result.iloc[0]["features_json"])
print("\nExtracted features:")
for k, v in out.items():
    print(f"  {k}: {v}")

# Contract assertions
REQUIRED_KEYS = {
    "claim_id", "claim_num", "lob_type", "loss_cause", "sub_type", "state",
    "jurisdiction_state", "water_source",
    "loss_date", "reported_date", "days_loss_to_report", "property_age_at_loss",
    "n_exposures", "has_dwelling", "has_contents", "has_living_expenses", "exposure_types",
    "desc_char_count", "docs_char_count", "notes_char_count", "incidents_char_count",
    "attorney_mentioned", "attorney_phrase_count", "siu_red_flag_count",
    "medical_severity_flag", "subrogation_opportunity_flag", "health_disclosure_flag",
    "urgency_score", "max_dollar_amount_mentioned", "dollar_amount_count",
    "feature_version", "extraction_errors",
}
missing = REQUIRED_KEYS - out.keys()
assert not missing, f"Missing feature keys: {missing}"
assert out["extraction_errors"] == [], f"Extraction errors: {out['extraction_errors']}"
print("\nContract OK — all required keys present, no extraction errors.")

/databricks/python/lib/python3.12/site-packages/mlflow/utils/annotations.py:303: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  return func(*args, **kwargs)


Testing payload: SYN-000

Extracted features:
  desc_char_count: 66
  docs_char_count: 733
  notes_char_count: 1155
  incidents_char_count: 161
  n_exposures: 3
  has_dwelling: True
  has_contents: True
  has_living_expenses: True
  exposure_types: ['Dwelling', 'Content', 'LivingExpenses']
  lob_type: HomeownersLine_HOE
  loss_cause: Water_Ext
  sub_type: WaterAndFreezing_Ext
  state: OH
  jurisdiction_state: OH
  water_source: other
  claim_id: ccp:10000000
  claim_num: 10000000-1
  loss_date: 2025-03-12
  reported_date: 2025-03-13
  days_loss_to_report: 1
  property_age_at_loss: 62
  attorney_mentioned: True
  attorney_phrase_count: 4
  siu_red_flag_count: 0
  medical_severity_flag: False
  subrogation_opportunity_flag: False
  health_disclosure_flag: False
  urgency_score: 0.1641
  max_dollar_amount_mentioned: None
  dollar_amount_count: 0
  feature_version: 0.1.0
  extraction_errors: []

Contract OK — all required keys present, no extraction errors.
